# 02 - Filtro de Hamilton e Decomposicao de Beveridge-Nelson

Neste notebook, exploramos duas alternativas modernas ao filtro HP:

1. **Filtro de Hamilton (2018)** - baseado em regressao, sem ciclos espurios
2. **Decomposicao de Beveridge-Nelson (1981)** - para series I(1), baseado em ARMA

### Conteudo
- Teoria e motivacao do filtro de Hamilton
- Implementacao e escolha de parametros ($h$, $p$)
- Decomposicao BN: tendencia permanente + ciclo transitorio
- Comparacao Hamilton vs HP vs BN
- Exercicios praticos

### Referencias
- Hamilton, J.D. (2018). *Why You Should Never Use the Hodrick-Prescott Filter*. Review of Economics and Statistics, 100(5), 831-843.
- Beveridge, S. & Nelson, C.R. (1981). *A New Approach to Decomposition of Economic Time Series into Permanent and Transitory Components*. Journal of Monetary Economics, 7(2), 151-174.
- Hodrick, R.J. & Prescott, E.C. (1997). *Postwar U.S. Business Cycles*. Journal of Money, Credit, and Banking.

## 1. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# Adicionar o diretorio raiz do projeto ao path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Imports do chronobox
from chronobox.filters import (
    hp_filter,
    hamilton_filter,
    hamilton_filter_detailed,
    bn_decomposition,
    bn_decomposition_detailed,
)

# Helpers
sys.path.insert(0, os.path.join(project_root, 'examples', 'filters'))
from utils.plot_helpers import plot_trend_cycle, plot_bandpass_comparison
from utils.data_generators import generate_trend_cycle

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Setup completo!')

In [ ]:
# Carregar PIB real dos EUA (trimestral)
data_dir = os.path.join(project_root, 'examples', 'filters', 'data')
us_gdp = pd.read_csv(os.path.join(data_dir, 'us_gdp_quarterly.csv'), parse_dates=['date'])

y = us_gdp['gdp_log'].values
dates = us_gdp['date']

print(f'PIB EUA: {len(y)} observacoes trimestrais')
print(f'Periodo: {dates.iloc[0].date()} a {dates.iloc[-1].date()}')

## 2. Filtro de Hamilton (2018)

### Motivacao

Hamilton (2018) propoe substituir o filtro HP por uma **regressao simples**:

$$y_t = \beta_0 + \beta_1 y_{t-h} + \beta_2 y_{t-h-1} + \cdots + \beta_p y_{t-h-p+1} + v_t$$

onde:
- $h$ = horizonte de previsao (8 trimestres = 2 anos para dados trimestrais)
- $p$ = numero de lags na regressao (4 para dados trimestrais)

A **tendencia** sao os valores ajustados ($\hat{y}_t$) e o **ciclo** sao os residuos ($v_t$).

### Vantagens sobre o HP

1. **Sem ciclos espurios**: Hamilton demonstra que o HP gera ciclicidade onde nao existe
2. **Sem problemas de endpoint**: os ultimos pontos nao distorcem a tendencia
3. **Fundamentacao estatistica**: baseado em um modelo de regressao interpretavel
4. **Robusto a raiz unitaria**: funciona tanto para series I(0) quanto I(1)

### Parametros recomendados

| Frequencia | $h$ | $p$ | Interpretacao |
|:-----------|----:|----:|:--------------|
| Trimestral | 8   | 4   | Ciclo de 2 anos, 1 ano de lags |
| Mensal     | 24  | 12  | Ciclo de 2 anos, 1 ano de lags |
| Anual      | 2   | 1   | Ciclo de 2 anos, 1 lag |

In [ ]:
# Hamilton filter com parametros recomendados: h=8, p=4
ham_trend, ham_cycle = hamilton_filter(y, h=8, p=4)

# Contagem de NaNs (primeiras h+p-1 obs sao NaN)
n_nan = np.sum(np.isnan(ham_cycle))
print(f'Hamilton Filter (h=8, p=4)')
print(f'  Obs com NaN: {n_nan} (primeiras h+p-1 = {8+4-1})')
print(f'  Obs validas: {len(y) - n_nan}')
print(f'  Desvio padrao do ciclo: {np.nanstd(ham_cycle):.4f}')

In [ ]:
# Resultados detalhados com coeficientes da regressao
ham_detailed = hamilton_filter_detailed(y, h=8, p=4)

print('Coeficientes da regressao de Hamilton:')
print(f'  Intercepto (beta_0): {ham_detailed.coefficients[0]:.4f}')
for i in range(1, len(ham_detailed.coefficients)):
    print(f'  beta_{i} (y_{{t-{ham_detailed.h + i - 1}}}): {ham_detailed.coefficients[i]:.4f}')
print(f'\n  h = {ham_detailed.h}, p = {ham_detailed.p}')

In [ ]:
# Grafico 1: Decomposicao de Hamilton
valid = ~np.isnan(ham_cycle)

fig = plot_trend_cycle(
    dates[valid], y[valid], ham_trend[valid], ham_cycle[valid],
    title='Filtro de Hamilton (h=8, p=4) - PIB Real EUA (log)'
)
plt.show()

### 2.1 Hamilton vs HP: Comparacao Direta

Vamos comparar o ciclo estimado pelo Hamilton com o ciclo do HP:

In [ ]:
# HP filter para comparacao
hp_trend, hp_cycle = hp_filter(y, lamb=1600)

# Grafico 2: Hamilton vs HP
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Tendencias
axes[0].plot(dates, y, alpha=0.4, label='PIB (log)', linewidth=0.8)
axes[0].plot(dates, hp_trend, label='Tendencia HP (λ=1600)', linewidth=1.5)
axes[0].plot(dates[valid], ham_trend[valid], label='Tendencia Hamilton (h=8)', linewidth=1.5, linestyle='--')
axes[0].set_title('Tendencia: HP vs Hamilton')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ciclos
axes[1].plot(dates, hp_cycle, label='Ciclo HP', alpha=0.7, linewidth=1)
axes[1].plot(dates[valid], ham_cycle[valid], label='Ciclo Hamilton', alpha=0.7, linewidth=1)
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1].set_title('Ciclo: HP vs Hamilton')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlacao (no intervalo valido)
corr = np.corrcoef(hp_cycle[valid], ham_cycle[valid])[0, 1]
print(f'Correlacao entre ciclo HP e Hamilton: {corr:.4f}')
print(f'Std HP: {hp_cycle.std():.4f}  |  Std Hamilton: {np.nanstd(ham_cycle):.4f}')

### 2.2 Sensibilidade ao Horizonte $h$

O parametro $h$ determina o horizonte da regressao e afeta diretamente
que tipo de flutuacoes sao capturadas como "ciclo".

In [ ]:
# Grafico 3: Diferentes horizontes h
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, h_val in enumerate([4, 8, 12, 16]):
    t_h, c_h = hamilton_filter(y, h=h_val, p=4)
    v = ~np.isnan(c_h)
    
    axes[i].plot(dates[v], c_h[v], linewidth=1.2, color='tab:blue')
    axes[i].axhline(0, color='black', linewidth=0.5, linestyle='--')
    axes[i].fill_between(dates[v], c_h[v], 0, alpha=0.2)
    axes[i].set_title(f'Hamilton (h={h_val}, p=4) - std={np.nanstd(c_h):.4f}')
    axes[i].grid(True, alpha=0.3)

fig.suptitle('Sensibilidade ao Horizonte h no Filtro de Hamilton', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Decomposicao de Beveridge-Nelson (BN)

### Teoria

A decomposicao BN (1981) e desenhada para series **I(1)** (com raiz unitaria).
Ela decompoe a serie em:

$$y_t = \tau_t + c_t$$

onde:
- $\tau_t$ = componente **permanente** (tendencia estocastica, passeio aleatorio com drift)
- $c_t$ = componente **transitorio** (ciclo estacionario)

### Procedimento

1. Tomar primeiras diferencas: $\Delta y_t = y_t - y_{t-1}$
2. Ajustar um modelo ARMA($p$, $q$) a $\Delta y_t$
3. Computar o multiplicador de longo prazo: $\psi(1) = \frac{\theta(1)}{\phi(1)}$
4. Decompor usando a representacao MA($\infty$)

O componente permanente captura choques com efeito duradouro;
o transitorio captura flutuacoes que se dissipam.

In [ ]:
# BN decomposition com AR(2) nas primeiras diferencas
bn_trend, bn_cycle = bn_decomposition(y, p=2, q=0)

print('Decomposicao de Beveridge-Nelson (AR(2))')
print(f'  Obs: {len(bn_trend)}')
print(f'  Std tendencia: {bn_trend.std():.4f}')
print(f'  Std ciclo: {bn_cycle.std():.4f}')

In [ ]:
# Resultados detalhados
bn_detailed = bn_decomposition_detailed(y, p=2, q=0)

print('Decomposicao BN - Detalhes:')
print(f'  Coeficientes AR: {bn_detailed.ar_coeffs}')
print(f'  Drift (media de Delta y): {bn_detailed.drift:.6f}')
print(f'  Multiplicador de longo prazo psi(1): {bn_detailed.psi_one:.4f}')
print(f'  Interpretacao: um choque unitario tem efeito permanente de {bn_detailed.psi_one:.4f}')

In [ ]:
# Grafico 4: Decomposicao BN
fig = plot_trend_cycle(
    dates, y, bn_trend, bn_cycle,
    title='Decomposicao de Beveridge-Nelson - PIB Real EUA (log)'
)
plt.show()

## 4. Comparacao: Hamilton vs HP vs BN

Vamos comparar os tres metodos lado a lado para o PIB dos EUA:

In [ ]:
# Grafico 5: Comparacao de tendencias
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Painel 1: Tendencias
axes[0].plot(dates, y, alpha=0.3, label='PIB (log)', linewidth=0.8, color='gray')
axes[0].plot(dates, hp_trend, label='HP (λ=1600)', linewidth=1.5)
axes[0].plot(dates[valid], ham_trend[valid], label='Hamilton (h=8)', linewidth=1.5, linestyle='--')
axes[0].plot(dates, bn_trend, label='BN (AR(2))', linewidth=1.5, linestyle=':')
axes[0].set_title('Comparacao de Tendencias: HP vs Hamilton vs BN')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Painel 2: Ciclos
axes[1].plot(dates, hp_cycle, label='HP', alpha=0.7, linewidth=1)
axes[1].plot(dates[valid], ham_cycle[valid], label='Hamilton', alpha=0.7, linewidth=1)
axes[1].plot(dates, bn_cycle, label='BN', alpha=0.7, linewidth=1)
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1].set_title('Comparacao de Ciclos: HP vs Hamilton vs BN')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Tabela resumo
print('Resumo estatistico dos componentes ciclicos:')
print(f'{"Filtro":<15} {"Std":>8} {"Min":>8} {"Max":>8} {"Corr c/ HP":>12}')
print('-' * 55)

for name, cycle in [('HP (λ=1600)', hp_cycle), ('Hamilton (h=8)', ham_cycle), ('BN (AR(2))', bn_cycle)]:
    v = ~np.isnan(cycle) & ~np.isnan(hp_cycle)
    corr = np.corrcoef(hp_cycle[v], cycle[v])[0, 1]
    print(f'{name:<15} {np.nanstd(cycle):>8.4f} {np.nanmin(cycle):>8.4f} {np.nanmax(cycle):>8.4f} {corr:>12.4f}')

## 5. Sensibilidade da Decomposicao BN a Ordem AR

A decomposicao BN depende da especificacao ARMA para as primeiras diferencas.
Vamos ver como a ordem AR afeta os resultados:

In [ ]:
# Grafico 6: BN com diferentes ordens AR
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, p_val in enumerate([1, 2, 4, 8]):
    bn_t, bn_c = bn_decomposition(y, p=p_val, q=0)
    
    axes[i].plot(dates, bn_c, linewidth=1.2, color='tab:purple')
    axes[i].axhline(0, color='black', linewidth=0.5, linestyle='--')
    axes[i].fill_between(dates, bn_c, 0, alpha=0.2, color='tab:purple')
    axes[i].set_title(f'BN (AR({p_val})) - std={bn_c.std():.4f}')
    axes[i].grid(True, alpha=0.3)

fig.suptitle('Sensibilidade da Decomposicao BN a Ordem AR', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Exercicio 1: Hamilton Filter para o Brasil

1. Carregue `brazil_gdp.csv` e aplique o filtro de Hamilton com $h=8$, $p=4$
2. Compare com o filtro HP ($\lambda=1600$)
3. Identifique os periodos de recessao no ciclo
4. Os dois filtros concordam sobre timing das recessoes?

In [ ]:
# Exercicio 1: Hamilton vs HP para PIB do Brasil
brazil_gdp = pd.read_csv(os.path.join(data_dir, 'brazil_gdp.csv'), parse_dates=['date'])

y_br = brazil_gdp['gdp_log'].values
dates_br = brazil_gdp['date']

# Hamilton
ham_t_br, ham_c_br = hamilton_filter(y_br, h=8, p=4)
# HP
hp_t_br, hp_c_br = hp_filter(y_br, lamb=1600)

valid_br = ~np.isnan(ham_c_br)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(dates_br, y_br, alpha=0.4, label='PIB Brasil (log)', linewidth=0.8)
axes[0].plot(dates_br, hp_t_br, label='Tendencia HP', linewidth=1.5)
axes[0].plot(dates_br[valid_br], ham_t_br[valid_br], label='Tendencia Hamilton', linewidth=1.5, linestyle='--')
axes[0].set_title('Tendencia: HP vs Hamilton - PIB Brasil')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(dates_br, hp_c_br, label='Ciclo HP', alpha=0.7)
axes[1].plot(dates_br[valid_br], ham_c_br[valid_br], label='Ciclo Hamilton', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1].set_title('Ciclo: HP vs Hamilton - PIB Brasil')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

corr_br = np.corrcoef(hp_c_br[valid_br], ham_c_br[valid_br])[0, 1]
print(f'Correlacao HP vs Hamilton (Brasil): {corr_br:.4f}')

## Exercicio 2: Comparando Diferentes Horizontes h no Hamilton

1. Aplique o filtro Hamilton ao PIB EUA com $h = 4, 8, 12, 16$ (mantendo $p=4$)
2. Compare os ciclos resultantes
3. Qual horizonte captura melhor os ciclos de negocios de 6-32 trimestres?
4. Como $h$ afeta a variancia do ciclo?

In [ ]:
# Exercicio 2: Diferentes horizontes
fig, ax = plt.subplots(figsize=(14, 5))

for h_val in [4, 8, 12, 16]:
    _, c_h = hamilton_filter(y, h=h_val, p=4)
    v = ~np.isnan(c_h)
    ax.plot(dates[v], c_h[v], label=f'h={h_val} (std={np.nanstd(c_h):.4f})', linewidth=1.1)

ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_title('Hamilton Filter: Efeito do Horizonte h')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercicio 3: BN para Dados Sinteticos

Aplique a decomposicao BN a dados sinteticos com ciclo conhecido:

1. Gere uma serie com `generate_trend_cycle(n=200, cycle_period=32, seed=42)`
2. Aplique BN com AR(2) e AR(4)
3. Compare o ciclo BN estimado com o ciclo verdadeiro
4. A BN recupera bem o ciclo? Por que pode ser diferente?

In [ ]:
# Exercicio 3: BN em dados sinteticos
synth = generate_trend_cycle(n=200, trend_type='linear', cycle_period=32, seed=42)
y_synth = synth['observed'].values
true_cycle = synth['cycle'].values
d_synth = synth['date']

bn_t2, bn_c2 = bn_decomposition(y_synth, p=2)
bn_t4, bn_c4 = bn_decomposition(y_synth, p=4)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(d_synth, true_cycle, 'k--', label='Ciclo verdadeiro', alpha=0.5)
axes[0].plot(d_synth, bn_c2, label='BN AR(2)', linewidth=1.2)
axes[0].set_title('BN AR(2) vs Ciclo Verdadeiro')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(d_synth, true_cycle, 'k--', label='Ciclo verdadeiro', alpha=0.5)
axes[1].plot(d_synth, bn_c4, label='BN AR(4)', linewidth=1.2, color='tab:orange')
axes[1].set_title('BN AR(4) vs Ciclo Verdadeiro')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Correlacao com ciclo verdadeiro:')
print(f'  BN AR(2): {np.corrcoef(bn_c2, true_cycle)[0,1]:.4f}')
print(f'  BN AR(4): {np.corrcoef(bn_c4, true_cycle)[0,1]:.4f}')

## 6. Conclusoes

- O **filtro de Hamilton** e uma alternativa robusta ao HP, baseada em regressao
  - Sem ciclos espurios, sem problemas de endpoint
  - O horizonte $h$ determina que tipo de flutuacoes sao capturadas
  - Recomendacao: $h=8$, $p=4$ para dados trimestrais

- A **decomposicao de Beveridge-Nelson** e desenhada para series I(1)
  - Separa choques permanentes (tendencia) de transitorios (ciclo)
  - Depende da especificacao ARMA; AR(2) geralmente e suficiente
  - O multiplicador $\psi(1)$ indica o efeito permanente de choques

- Para ciclos de negocios, HP e Hamilton concordam sobre timing,
  mas Hamilton e preferivel por sua fundamentacao estatistica

No proximo notebook, exploraremos o **Spillover Index de Diebold-Yilmaz**
para analisar interdependencias entre series.